# Active Learning with GLiNER for NER and Relation Extraction


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install gliner accelerate
!pip install --upgrade datasets huggingface_hub

# Process

1. define a manual annotated holdout (holdout_data)
2. train the base multitask model with an initial train set.
3. check performance
4. introduce new documents to the pipeline (min 5)
  * infer on these new sentences using thresh=0.5 (or lower)
  * revalidate the predictions to make it high quality
  * the revalidated predictions (pred_reval) will be used to create synthetic data to different domains -> append to the initial data
  * retrain
  * check performance to the holdout data

end

Helper for restructuring annotated data to predictions and relations (for simplicity of generating synthetic data)

In [ ]:
import re

def ner_normalized_to_pipeline(ner_normalized, tokenized_text, text):
    """
    Convert ner_normalized (list of [token_start, token_end, payload])
    back to 'predictions' and 'relations' with character spans.

    Args:
        ner_normalized: List of [ts, te, payload]
            payload either 'label' or 'head <> relation'
        tokenized_text: List of tokens
        text: Full original text string

    Returns:
        predictions: list of dicts with keys 'start','end','text','label'
        relations: dict mapping head_text -> list of dicts
                   each with 'source','relation','target','start','end'
    """
    # 1) Build token offsets
    offsets = []
    ptr = 0
    for tok in tokenized_text:
        idx = text.find(tok, ptr)
        if idx < 0:
            raise ValueError(f"Token {tok!r} not found after {ptr}")
        offsets.append((idx, idx + len(tok)))
        ptr = idx + len(tok)

    def token_to_char_span(ts, te):
        """Convert token span [ts,te] inclusive to char start/end."""
        start_char = offsets[ts][0]
        end_char = offsets[te][1]
        return start_char, end_char

    predictions = []
    relations = {}

    for ts, te, payload in ner_normalized:
        start_char, end_char = token_to_char_span(ts, te)
        if '<>' in payload:
            head_text, relation = [p.strip() for p in payload.split('<>')]
            # For relations, target_text is the text span
            target_text = text[start_char:end_char]
            relations.setdefault(head_text, []).append({
                'source':   head_text,
                'relation': relation,
                'target':   target_text,
                'start':    start_char,
                'end':      end_char
            })
        else:
            label = payload.strip()
            entity_text = text[start_char:end_char]
            predictions.append({
                'start': start_char,
                'end':   end_char,
                'text':  entity_text,
                'label': label
            })

    return predictions, relations

Helper functions for revalidating predictions and for generating synthetic data

In [ ]:
import re
def llm_revalidate_with_relations(doc):
    """
    Revalidates dataset entity predictions and their relations via LLM.

    Args:
        doc (dict): Must contain keys 'text', 'predictions', and 'relations'.
            - 'text' (str): Document passage.
            - 'predictions' (list): List of dicts with keys 'start','end','text','label','score'.
            - 'relations' (dict): Mapping from prediction text to list of dicts with keys
              'source','relation','target','score','start','end'.

    Returns:
        dict: JSON with exactly two keys:
            - 'predictions': original predictions enriched with 'action' ('keep' or 'remove') and 'reason'.
            - 'relations': same structure as input, each relation enriched with 'action' and 'reason'.

    Raises:
        ValueError: If LLM output is not valid JSON or missing required keys.
    """
    prompt = f"""
Definition:
- A dataset is a structured collection of data used for research or analysis (e.g., indexes, registry, literature, study/studies, systems, surveys, indicators, databases).
- Non-datasets are texts or concepts that should NOT be treated as datasets (e.g., organizations & institutions, policy documents, economic models & frameworks, legislation & agreements).

### Relation Type Definitions

- **acronym**
  The short‐form name or abbreviation of a dataset (e.g., “DHS” for “Demographic and Health Survey”).

- **author**
  The person or team that collected or assembled the dataset (e.g., “World Bank Research Group”).

- **data description**
  A brief textual description of what the dataset contains (e.g., “household income microdata from 2018”).

- **data geography**
  The geographic area covered by the dataset (e.g., “Kenya”, “Sub-Saharan Africa”).

- **data source**
  The original platform or instrument used to collect the dataset (e.g., “Kenya COVID-19 Rapid Response Phone Survey”, “UN Comtrade database”). **Do not** use this for thematic words like “COVID-19” alone, which describe an event or condition, not the collection instrument itself.

- **data type**
  The technical format of the data (e.g., “microdata”, “time-series”, “panel data”).

- **publication year**
  The year the dataset was first released or published.

- **publisher**
  The organization or agency that formally disseminated the dataset (e.g., “UNHCR”, “World Bank”).

- **reference population**
  The population group to which the dataset pertains (e.g., “urban refugee households”, “Kenyan nationals”).

- **reference year**
  The time period the data describe (e.g., “2020–2021 survey wave”).

- **version**
  The dataset’s version identifier, if any (e.g., “v2.0”, “2018 release”).

---

Task: Given the passage, 'predictions', and 'relations' in the exact input format:
1. **Validate & reclassify each prediction**
   - Duplicate entries are okay.
   - Every prediction must end up with one of:
     - **named dataset**: A formally titled resource (e.g. “Demographic and Health Survey”).
     - **unnamed dataset**: A descriptive mention of data or data collection (“household survey”, “longitudinal panel data”, “school-based health survey”).
     - **vague dataset**: Extremely generic references of data or data collection (e.g. “data”, “records”, “systems”, “literature”, “indexes”) without clear collection context.
   - **be forgiving on unnamed and vague dataset mentions** if it is used in the context or alongside other data or data collection `keep` it only `remove` if it is clearly not a data collection or a dataset (organizations, models, etc.).
Return the same 'predictions' list, adding keys 'action' and 'reason' to each entry.

- For each relation under 'relations', validate link between a kept prediction and its attribute:
    • 'keep' if correct, vague and unnamed dataset or data can have relations as long as they are correct.
    • 'remove' if incorrect or refers to a removed prediction.
Return the same 'relations' dict, adding 'action' and 'reason' to each relation dict.

Do NOT add new datasets or relations. Do NOT modify any existing keys, order, or text. Return only valid JSON.

Passage:
{doc['text']}

predictions:
{json.dumps(doc['predictions'], indent=2)}

relations:
{json.dumps(doc['relations'], indent=2)}
"""

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=10000
    )
    content = response.choices[0].message.content

    # Strip markdown fences
    m = re.match(r"```(?:json)?\n([\s\S]*?)```", content)
    if m:
        content = m.group(1).strip()

    # Try to parse, and on failure attempt a simple brace‐balance repair
    try:
        result = json.loads(content)
    except JSONDecodeError:
        # count braces
        opens  = content.count('{')
        closes = content.count('}')
        # if there are fewer closes, append the difference
        if opens > closes:
            content += '}' * (opens - closes)
            try:
                result = json.loads(content)
            except JSONDecodeError:
                raise ValueError(f"LLM output is not valid JSON even after repair:\n{content}")
        else:
            raise ValueError(f"LLM output is not valid JSON:\n{content}")

    # Validate structure
    if set(result.keys()) != {'predictions', 'relations'}:
        raise ValueError("Output JSON must contain exactly 'predictions' and 'relations'.")

    return result

In [ ]:
def build_validated_ner_char(preds, rels):
    """
    Merge validated predictions and relations into character-level NER format.
    Returns list of [start, end, payload_string].
    """
    payloads = []
    # 1) Add each kept prediction with label only
    for p in preds:
        payloads.append([p['start'], p['end'], p['label']])
    # 2) Add each kept relation using its own span, label as source <> relation
    for src, rel_list in rels.items():
        for r in rel_list:
            payloads.append([r['start'], r['end'], f"{r['source']} <> {r['relation']}"])
    # 3) Sort by start then end
    payloads.sort(key=lambda x: (x[0], x[1]))
    return payloads

In [ ]:
import os
import json
import uuid
from json import JSONDecodeError
from tqdm.auto import tqdm

def revalidate_and_backup(
    docs,
    llm_revalidate_fn,
    build_ner_fn,
    backup_dir="backup_results"
):
    """
    Revalidates model predictions and relations via an LLM, filters for 'keep' actions,
    saves each result as a backup JSON, and returns the list of results.

    Args:
        docs (list[dict]): Each dict must have 'filename', 'page', 'text', 'predictions', 'relations'.
        llm_revalidate_fn (callable): Function(doc)->{'predictions','relations'} from LLM.
        build_ner_fn (callable): Function(preds, rels)->ner_text_validated payload.
        backup_dir (str): Directory to save backup JSON files.

    Returns:
        list[dict]: List of all result entries (with validated fields).
    """
    os.makedirs(backup_dir, exist_ok=True)
    results = []

    for doc in tqdm(docs, desc="Revalidating using LLM"):
        filename = doc.get("filename", "unknown")
        page = doc.get("page", "NA")

        if not doc.get("predictions"):
            continue

        try:
            validated = llm_revalidate_fn({
                'text': doc['text'],
                'predictions': doc['predictions'],
                'relations': doc['relations']
            })

            # Filter only 'keep'
            #print(validated)
            kept_preds = [p for p in validated['predictions'] if p.get('action') != 'remove']
            kept_rels = {
                src: [r for r in rels if r.get('action') != 'remove']
                for src, rels in validated['relations'].items()
                if any(r.get('action') != 'remove' for r in rels)
            }

        except (ValueError, JSONDecodeError, Exception) as e:
            print(f"[Validation skipped for {filename}]: {e}")
            continue

        # Build the result entry
        entry = {
            'filename': filename,
            'page': page,
            'text': doc['text'],
            'predictions': doc['predictions'],
            'relations': doc['relations'],
            'predictions_validated_raw': validated['predictions'],
            'relations_validated_raw': validated['relations'],
            'predictions_validated': kept_preds,
            'relations_validated': kept_rels,
            'ner_text_validated': build_ner_fn(kept_preds, kept_rels),
        }
        results.append(entry)

        # Save backup
        backup_fname = f"{filename}_page{page}_{uuid.uuid4().hex}.json"
        backup_path = os.path.join(backup_dir, backup_fname)
        with open(backup_path, "w", encoding="utf-8") as f:
            json.dump(entry, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(results)} entries to '{backup_dir}/'")
    return results

In [ ]:
# Domain list
domains = [
    "meteorology & weather modeling",
    "oceanography & marine science",
    "environmental science",
    "hydrology & water resource management",
    "biodiversity & ecosystem analysis",
    "air pollution & emissions monitoring",
    "energy & sustainability studies",
    "macroeconomics & development studies",
    "sustainable finance & green investment",
    "urban development & city planning",
    "disaster response & climate resilience",
    "education & human capital development",
    "social protection & welfare programs",
    "infrastructure development & transport economics",
    "sustainable development goals (SDGs) monitoring",
]


def generate_synthetic_data(entry: dict, n: int = 5) -> list:
    """
    Given a single validated entry, generate n synthetic examples.
    Each output will have keys 'text', 'predictions', and 'relations'.
    """
    # Prepare the minimal JSON for the prompt (dropping spans)
    data = {
        "text": entry["text"],
        "predictions": [
            {"text": p["text"], "label": p["label"]}
            for p in entry["predictions_validated"]
        ],
        "relations": {
            src: [
                {"source": r["source"], "relation": r["relation"], "target": r["target"]}
                for r in rels
            ]
            for src, rels in entry["relations_validated"].items()
        }
    }


    # Randomly select a domain for this generation
    domain = random.choice(domains)

    prompt = (
        f"Context Domain: {domain}\n"
        "Given the following JSON data, generate a list of "
        f"{n} different comprehensive texts. This should only return valid JSON as a list of "
        "dictionaries. Do not say anything else. Each description should mimic the structure of "
        "the original input:\n\n"
        "JSON Data:\n"
        + json.dumps(data, indent=2)
        + "\n\n"
        f"Generate {n} different examples in JSON format that follow the structure and content "
        "of the provided data. Each example must have keys: 'text', 'predictions', 'relations'.\n"
        "**Instructions** \n"
        "1. the 'predictions' text should match how it was mentioned in the input text sentence"
        "2. the relations should match what was in the input 'relations'"
        "3. the 'source' should match what was predicted"
    )

    # Call the API
    resp = client.chat.completions.create(
        model=SYNTH_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=10000
    )

    content = resp.choices[0].message.content

    # Strip any ``` fences
    fence = re.match(r"```(?:json)?\n([\s\S]*?)```", content)
    if fence:
        content = fence.group(1).strip()

    # parse
    try:
        output = json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse LLM output as JSON:\n{content}") from e

    return output

## Model Training
Now that we have the above, we can now loop in training.
First, we train with the initial train data and the synthetic data generated.

In [ ]:
import os
import json
import uuid

def generate_and_backup_synthetic(
    rev_results: list,
    generate_synthetic_data_fn,
    base_dir: str,
    n: int = 5
) -> list:
    """
    For each validated result in rev_results, generate synthetic data,
    save it under base_dir/<filename>/synthetic_<uuid>.json, and
    return the list of full result entries including the synthetic data.

    Args:
        rev_results: List of dicts, each must contain 'filename'.
        generate_synthetic_data_fn: Function(entry, n) -> List[synthetic examples].
        base_dir: Root directory to save synthetic backups.
        n: Number of synthetic examples per entry.

    Returns:
        List of dicts: original rev_results entries augmented with
        a 'synthetic' key containing the generated examples.
    """
    os.makedirs(base_dir, exist_ok=True)
    all_augmented = []

    for entry in tqdm(rev_results, desc="Generating Synthetic Data"):
        filename = entry.get("filename", str(uuid.uuid4()))
        # Generate synthetic examples
        try:
          synthetic = generate_synthetic_data_fn(entry, n=n)
        except Exception as e:
          print(f'skipping because of an error {e}')
          continue

        # Augment entry
        augmented = entry.copy()
        augmented["synthetic"] = synthetic
        all_augmented.append(augmented)

        # Save backup
        folder = os.path.join(base_dir, filename)
        os.makedirs(folder, exist_ok=True)
        backup_fname = f"synthetic_{uuid.uuid4().hex}.json"
        backup_path = os.path.join(folder, backup_fname)
        with open(backup_path, "w", encoding="utf-8") as bf:
            json.dump(synthetic, bf, ensure_ascii=False, indent=2)

    return all_augmented

In [ ]:
import os
import json
from glob import glob
from typing import List

def load_synthetic_results(root_dir: str) -> List[dict]:
    """
    Walks through all subfolders under `root_dir` and loads every JSON file,
    returning a flat list of dicts (each parsed from one JSON file).

    Example directory structure:
      root_dir/
        IDU-xxxx/
          synthetic_a1b2c3.json
          synthetic_d4e5f6.json
        IDU-yyyy/
          synthetic_123abc.json

    Returns:
        List of all JSON‐loaded objects from those files.
    """
    results = []
    # find all json files under any immediate subfolder
    pattern = os.path.join(root_dir, '*', '*.json')
    for path in tqdm(glob(pattern), desc="loading synthetic"):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                results.append(json.load(f))
        except Exception as e:
            print(f"Warning: failed to load {path}: {e}")
    return results

In [ ]:
import os
import json
from glob import glob
from typing import List, Any

def load_and_flatten_revalidated(root_dir: str) -> List[Any]:
    """
    Loads every JSON file in *root_dir* (and its subfolders), each file containing
    one revalidated entry, and returns one flat list of all entries.
    """
    flat_results: List[Any] = []
    # Match any .json file under root_dir or its subdirectories
    pattern = os.path.join(root_dir, '**', '*.json')
    for path in tqdm(glob(pattern, recursive=True), desc="loading revalidated data"):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Each file is a single dict, append directly
                if isinstance(data, dict):
                    flat_results.append(data)
                else:
                    print(f"Warning: {path} did not contain a JSON object, skipping.")
        except Exception as e:
            print(f"Warning: failed to load {path}: {e}")
    return flat_results


Prepare revalidated data for training

In [ ]:
import os
import json
def load_json_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The specified file does not exist: {file_path}")
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

In [ ]:
import re

def tokenize_text(text):
    """Tokenize the input text into a list of tokens."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

In [ ]:
from typing import List, Dict, Any, Tuple
from collections import defaultdict

def compute_token_offsets(text: str, tokens: List[str]) -> List[Tuple[int,int]]:
    """
    For each token in `tokens`, find its (char_start, char_end) in `text`.
    """
    offsets = []
    ptr = 0
    for tok in tokens:
        idx = text.find(tok, ptr)
        if idx < 0:
            raise ValueError(f"Token {tok!r} not found in text after position {ptr}")
        offsets.append((idx, idx + len(tok)))
        ptr = idx + len(tok)
    return offsets

def char_to_token_span(
    char_start: int,
    char_end:   int,
    token_offsets: List[Tuple[int,int]]
) -> Tuple[int,int]:
    """
    Convert a character‐level span [char_start, char_end] (inclusive) into
    the smallest enclosing token index span [token_start, token_end_inclusive].
    Returns (None, None) if no tokens overlap.
    """
    # find all tokens whose span overlaps [char_start, char_end]
    toks = [
        i for i,(s,e) in enumerate(token_offsets)
        if not (e < char_start or s > char_end)
    ]
    if not toks:
        return None, None
    return min(toks), max(toks)

def build_entities_from_validated(doc: Dict[str,Any]) -> Dict[str,Any]:
    """
    Given a revalidated doc with:
      - 'text': full string
      - 'tokenized_text': List[str] of tokens
      - 'predictions': [ {start,char_end,text,label} ... ]
      - 'relations': { source_text: [ { source,relation,target,start,end } ... ] }
    Returns a dict:
      {
        'text': same text,
        'tokenized_text': same list of tokens,
        'entities': [
           { 'start': tok_start, 'end': tok_end, 'text': span_text, 'label': label },
           ...
        ]
      }
    where start/end are token‐level inclusive indices.
    """
    text   = doc['text']
    tokens = doc['tokenized_text']
    offsets = compute_token_offsets(text, tokens)

    entities = []

    # 1) predictions → entities
    for p in doc['predictions']:
        cs, ce = p['start'], p['end']
        ts, te = char_to_token_span(cs, ce, offsets)
        if ts is None:
            # couldn't align char span → skip or log
            continue
        entities.append({
            'text':  p['text'],
            'label': p['label'],
            'start': ts,
            'end':   te
        })

    # 2) relations → entities (as source <> relation)
    for src, rel_list in doc['relations'].items():
        for r in rel_list:
            cs, ce = r['start'], r['end']
            ts, te = char_to_token_span(cs, ce, offsets)
            if ts is None:
                continue
            entities.append({
                'text':  r['target'],
                'label': f"{r['source']} <> {r['relation']}",
                'start': ts,
                'end':   te
            })

    return {
        'text':           text,
        'tokenized_text': tokens,
        'entities':       entities
    }


In [ ]:
def convert_my_data_for_gliner_train(dataset):
    """
    Input `dataset` is a list of samples, each with:
      - sample["text"]: the full text (not used for span lookup here)
      - sample["entities"]: a list of dicts, each with:
            * "text": the mention string
            * "label": either a plain entity label or a relation label containing "<>"
            * "start": token‐level start index
            * "end": token‐level end index
    We build:
      - ents: all true entities (labels without "<>"), carrying over their start/end.
      - rels: for each item whose label contains "<>", split label into head_text <> rel_type,
              then look up head_text among the other entities to get head_start/head_end,
              and take tail_start/tail_end directly from this item.
    """
    out = []
    for sample in dataset:
        text = sample["text"]
        ents = []
        rels = []

        # collect all pure entities into a map by their text
        # so we can find head spans later.
        entity_map = {}
        for item in sample["entities"]:
            lbl = item["label"]
            ent_text = item["text"]
            if "<>" not in lbl:
                # plain entity
                start = item["start"]
                end = item["end"]
                ents.append({
                    "text": ent_text,
                    "label": lbl,
                    "start": start,
                    "end": end
                })
                # map entity text → (start, end)
                entity_map[ent_text] = (start, end)

        # handle relations
        for item in sample["entities"]:
            lbl = item["label"]
            ent_text = item["text"]
            if "<>" in lbl:
                # split "head_text <> rel_type"
                head_text, rel_type = lbl.split(" <> ", 1)
                head_text = head_text.strip()
                rel_type = rel_type.strip()

                # tail (this item's text) already has its own start/end
                tail_text = ent_text
                tail_start = item["start"]
                tail_end = item["end"]

                # find head span by looking up head_text in entity_map
                head_span = entity_map.get(head_text)
                if head_span is None:
                    # if head entity wasn’t in the pure-entity list, skip
                    continue
                head_start, head_end = head_span

                rels.append({
                    "head": head_text,
                    "head_start": head_start,
                    "head_end": head_end,
                    "tail": tail_text,
                    "tail_start": tail_start,
                    "tail_end": tail_end,
                    "label": rel_type
                })

        out.append({
            "text": text,
            "entities": ents,
            "relations": rels
        })

    return out

In [ ]:
def convert_gliner_output_to_token_level(gliner_data):
    """
    Input: gliner_data is the list returned by convert_my_data_for_gliner_train().
    Each element is a dict with:
      - "text": full raw text
      - "entities": list of dicts, each having:
            { "text", "label", "start": token_start, "end": token_end }
      - "relations": list of dicts, each having:
            {
              "head": head_text,
              "head_start": token_start,
              "head_end":   token_end,
              "tail": tail_text,
              "tail_start": token_start,
              "tail_end":   token_end,
              "label": rel_type
            }
    We produce for each sample a dict with:
      - "tokenized_text": List[str]
      - "ner": List of [entity_start, entity_end, entity_label]
      - "re":  List of [head_start, head_end, tail_start, tail_end, rel_label]
    """
    token_level = []

    for sample in gliner_data:
        text = sample["text"]

        # Re‐tokenize to have tokenized_text available to the model
        tokens, _ = tokenize_text_spans(text)

        # Build NER spans directly from sample["entities"]
        ner_spans = []
        for ent in sample.get("entities", []):
            # ent["start"] and ent["end"] are already token indices
            s_tok = ent["start"]
            e_tok = ent["end"]
            lbl   = ent["label"]
            ner_spans.append([s_tok, e_tok, lbl])

        # Build RE triples directly from sample["relations"]
        re_triples = []
        for rel in sample.get("relations", []):
            hs = rel["head_start"]
            he = rel["head_end"]
            ts = rel["tail_start"]
            te = rel["tail_end"]
            lbl = rel["label"]
            re_triples.append([hs, he, ts, te, lbl])

        token_level.append({
            "tokenized_text": tokens,
            "ner": ner_spans,
            "re": re_triples
        })

    return token_level

In [ ]:
def tokenize_text_spans(text):
    """
    Tokenize `text` into a list of tokens (words or punctuation) and return
    both the token list and a parallel list of (char_start, char_end) spans.
    We only need the tokens here, since all start/end indices are already token‐level.
    """
    pattern = re.compile(r"\w+(?:[-_]\w+)*|\S")
    matches = list(pattern.finditer(text))
    tokens = [m.group(0) for m in matches]
    spans  = [(m.start(), m.end()) for m in matches]
    return tokens, spans

In [ ]:
def process_revalidated_data(revalidated_data):

  this_data_train = pd.DataFrame(revalidated_data)
  this_data_train['tokenized_text'] = this_data_train['text'].apply(tokenize_text)
  this_data_train = this_data_train.loc[this_data_train['predictions_validated'].apply(lambda x: x != [])]
  this_data_train = this_data_train.rename(columns={'predictions': 'pred_old', 'relations': 'rel_old',
                              'predictions_validated': 'predictions', 'relations_validated': 'relations'})
  this_data_train = this_data_train[['text', 'tokenized_text','predictions', 'relations']].to_dict(orient='records')
  this_data_train = [build_entities_from_validated(doc) for doc in this_data_train]
  this_data_train = convert_my_data_for_gliner_train(this_data_train)
  this_data_train = convert_gliner_output_to_token_level(this_data_train)

  return this_data_train

Synthetic data processing

In [ ]:
import json

def transform_synthetic_to_entities(all_data_synth):
    """
    Transforms the synthetic generation output into the desired format:
    - Keeps 'text' as is.
    - Builds 'entities' list from 'predictions' and 'relations':
        * Each prediction -> {'text': ..., 'label': ...}
        * Each relation -> {'text': target, 'label': 'source <> relation'}
    """
    transformed = []
    for group in all_data_synth:
        for example in group:
            new_entry = {'text': example['text'], 'entities': []}
            # Add predictions
            for p in example.get('predictions', []):
                new_entry['entities'].append({
                    'text': p['text'],
                    'label': p['label']
                })
            # Add relations
            for src, rels in example.get('relations', {}).items():
                for r in rels:
                    new_entry['entities'].append({
                        'text': r['target'],
                        'label': f"{r['source']} <> {r['relation']}"
                    })
            transformed.append(new_entry)
    return transformed

In [ ]:
import json

def convert_my_data_for_gliner(dataset):
    out = []
    for sample in dataset:
        text = sample['text']
        ents, rels = [], []

        for item in sample['entities']:
            lbl = item['label']
            ent_text = item['text']
            if ' <>' in lbl:
                # this is a relation
                head_text, rel_type = lbl.split(' <> ', 1)
                tail_text = ent_text
                rels.append({
                    'head': head_text,
                    'tail': tail_text,
                    'label': rel_type
                })
            else:
                # this is an entity
                start = text.find(ent_text)
                if start == -1:
                    # fallback: skip or warn
                    continue
                end = start + len(ent_text)
                ents.append({
                    'text': ent_text,
                    'label': lbl,
                    'start': start,
                    'end': end
                })

        out.append({
            'text': text,
            'entities': ents,
            'relations': rels
        })
    return out

In [ ]:
def filter_entities_and_relations(docs, allowed_entity_labels, allowed_relation_types):
    """
    Given a list of document dicts with keys 'text', 'entities', and 'relations',
    returns a new list where:
      - 'entities' is filtered to only include those with 'label' in allowed_entity_labels
      - 'relations' is filtered to only include those with 'label' (or 'relation')
        in allowed_relation_types
    """
    filtered_docs = []
    for doc in docs:
        # Filter entities
        ents = [
            e for e in doc.get('entities', [])
            if e.get('label') in allowed_entity_labels
        ]
        # Filter relations (relation label under key 'relation' or 'label')
        rels = []
        for r in doc.get('relations', []):
            rel_label = r.get('relation', r.get('label'))
            if rel_label in allowed_relation_types:
                rels.append(r)

        filtered_docs.append({
            'text': doc.get('text'),
            'entities': ents,
            'relations': rels
        })
    return filtered_docs

In [ ]:
def convert_to_token_level_contextual(examples):
    token_level = []

    for ex in examples:
        text = ex["text"]
        # tokenize using regex helper
        tokens, offsets = tokenize_text_spans(text)

        # map character index to token index
        def char_to_token(cidx):
            for tidx, (s, e) in enumerate(offsets):
                if s <= cidx < e:
                    return tidx
            return None

        # build a map of chr‐level spans for explicitly annotated entities
        ent_char_spans = {
            ent["text"]: (ent["start"], ent["end"])
            for ent in ex.get("entities", [])
        }
        # convert entity char spans to token spans
        ent_token_spans = {}
        for txt, (s_char, e_char) in ent_char_spans.items():
            s_tok = char_to_token(s_char)
            e_tok = char_to_token(e_char - 1)
            if s_tok is not None and e_tok is not None:
                ent_token_spans[txt] = (s_tok, e_tok)

        # build NER list (only original entities)
        ner = []
        for ent in ex.get("entities", []):
            span = ent_token_spans.get(ent["text"])
            if span:
                s_tok, e_tok = span
                ner.append([s_tok, e_tok, ent["label"]])

        # build RE list using contextual matching for tail
        re_triples = []
        for rel in ex.get("relations", []):
            # HEAD: find head char span
            if rel["head"] in ent_char_spans:
                h_char_start, h_char_end = ent_char_spans[rel["head"]]
            else:
                # fallback: find first occurrence of head in text
                idx = text.find(rel["head"])
                if idx < 0:
                    continue
                h_char_start = idx
                h_char_end = idx + len(rel["head"])

            # map head char span -> token span
            h_s_tok = char_to_token(h_char_start)
            h_e_tok = char_to_token(h_char_end - 1)
            if h_s_tok is None or h_e_tok is None:
                continue

            # TAIL: find tail char span, but search starting after head_char_end
            if rel["tail"] in ent_char_spans:
                t_char_start, t_char_end = ent_char_spans[rel["tail"]]
            else:
                idx = text.find(rel["tail"], h_char_end)
                if idx < 0:
                    # if not found after head, fallback to global find
                    idx = text.find(rel["tail"])
                    if idx < 0:
                        continue
                    t_char_start = idx
                else:
                    t_char_start = idx
                t_char_end = t_char_start + len(rel["tail"])

            # map tail char span -> token span
            t_s_tok = char_to_token(t_char_start)
            t_e_tok = char_to_token(t_char_end - 1)
            if t_s_tok is None or t_e_tok is None:
                continue

            # append RE triple: [head_start, head_end, tail_start, tail_end, label]
            re_triples.append([h_s_tok, h_e_tok, t_s_tok, t_e_tok, rel["label"]])

        token_level.append({
            "tokenized_text": tokens,
            "ner": ner,
            "re": re_triples,
            #"ent_token_spans": ent_token_spans
        })

    return token_level

In [ ]:
def process_synthetic_data(all_data_synth):
  transformed_synth = transform_synthetic_to_entities(all_data_synth)
  transformed_synth = convert_my_data_for_gliner(transformed_synth)
  transformed_synth = [t for t in transformed_synth if t['entities'] != []]

  TARGET_ENTITY_LABELS = {"named dataset", "unnamed dataset", "vague dataset"}
  ALLOWED_RELATIONS = [
      'acronym', 'author', 'data description', 'data geography',
      'data source', 'data type', 'publication year', 'publisher',
      'reference population', 'reference year', 'version'
  ]

  transformed_synth = filter_entities_and_relations(transformed_synth, TARGET_ENTITY_LABELS, ALLOWED_RELATIONS)
  transformed_synth_token = convert_to_token_level_contextual(transformed_synth)
  return transformed_synth_token

In [ ]:
def make_train_test_split(token_level_data):

  os.makedirs("assets", exist_ok=True)
  split_idx = int(0.9 * len(token_level_data))

  train, test = token_level_data[:split_idx], token_level_data[split_idx:]

  return train, test

In [ ]:
import torch

def load_model(model_path):
  # Device and model setup
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = GLiNER.from_pretrained(model_path).to(device)

  return model

In [ ]:
from gliner.multitask.base import GLiNERBasePipeline
from typing import Optional, List, Union
from datasets import load_dataset, Dataset
from gliner import GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollator

In [ ]:
def train_gliner_model(
    synthetic_dataset,
    validated_dataset,
):

  # prefinetune
  model = load_model('knowledgator/gliner-multitask-v1.0')
  synth_train, synth_test = make_train_test_split(synthetic_dataset)
  data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)
  # calculate number of epochs
  num_steps = 500
  batch_size = 8
  data_size = len(synth_train)
  num_batches = data_size // batch_size
  num_epochs = max(1, num_steps // num_batches)

  training_args = TrainingArguments(
      output_dir="models",
      learning_rate=1e-5,
      weight_decay=0.01,
      others_lr=1e-5,
      others_weight_decay=0.01,
      lr_scheduler_type="cosine", #for prefinetuning
      warmup_ratio=0.1,
      per_device_train_batch_size=batch_size,
      per_device_eval_batch_size=batch_size,
      focal_loss_alpha=0.75,
      focal_loss_gamma=2,
      num_train_epochs=num_epochs,
      eval_strategy="steps",
      save_steps = 250,
      save_total_limit=10,
      dataloader_num_workers = 0,
      use_cpu = False,
      report_to="none",
      )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=synth_train,
      eval_dataset=synth_test,
      data_collator=data_collator,
  )

  trainer.train()


  ### finetuning

  validated_train, validated_test = make_train_test_split(validated_dataset)
  data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)

  # calculate number of epochs
  num_steps = 500
  batch_size = 4
  data_size = len(validated_train)
  num_batches = data_size // batch_size
  num_epochs = max(1, num_steps // num_batches)
  #num_epochs = 4
  training_args = TrainingArguments(
      output_dir="models",
      learning_rate=5e-6,
      weight_decay=0.01,
      others_lr=1e-5,
      others_weight_decay=0.01,
      lr_scheduler_type="linear", #fine tuning
      warmup_ratio=0.1,
      per_device_train_batch_size=batch_size,
      per_device_eval_batch_size=batch_size,
      focal_loss_alpha=0.75,
      focal_loss_gamma=2,
      num_train_epochs=num_epochs,
      eval_strategy="steps",
      save_steps = 250,
      save_total_limit=10,
      dataloader_num_workers = 0,
      use_cpu = False,
      report_to="none",
      )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=validated_train,
      eval_dataset=validated_test,
      data_collator=data_collator,
  )

  trainer.train()

  return model

In [ ]:
# created a custom GLiNER Extractor -> pulled request to GLiNER main and approved!
class CustomGLiNERRelationExtractor(GLiNERBasePipeline):
    """
    A class to use GLiNER for relation extraction inference and evaluation.

    Attributes:
        device (str): Device to run the model on, e.g., 'cuda:0' or 'cpu'.
        model (GLiNER): Loaded GLiNER model instance.
        prompt (str): Template prompt for relation extraction.

    Methods:
        process_predictions(predictions):
            Processes model predictions to extract the most likely labels.
        prepare_texts(texts, labels):
            Creates relation extraction prompts for each input text.
        __call__(texts, labels, threshold=0.5):
            Runs the model on the given texts and returns predicted labels.
        evaluate(dataset_id, labels=None, threshold=0.5, max_examples=-1):
            Evaluates the model on a dataset and computes F1 scores.
    """

    prompt = "Extract relationships between entities from the text: "

    def __init__(self, model_id: str = None, model: GLiNER = None, device: str = 'cuda:0', ner_threshold: float = 0.5, rel_threshold: float = 0.5, return_index: bool = False, prompt: Optional[str] = None):
        """
        Initializes the GLiNERRelationExtractor.

        Args:
            model_id (str, optional): Identifier for the model to be loaded. Defaults to None.
            model (GLiNER, optional): Preloaded GLiNER model. Defaults to None.
            device (str, optional): Device to run the model on ('cpu' or 'cuda:X'). Defaults to 'cuda:0'.
            ner_threshold (float, optional): Named Entity Recognition threshold to use. Defaults to 0.5.
            rel_threshold (float, optional): Relation Extraction threshold to use. Defaults to 0.5.
            prompt (str, optional): Template prompt for question-answering.
        """
        # Use the provided prompt or default to the class-level prompt
        prompt = prompt if prompt is not None else self.prompt
        self.return_index = return_index
        super().__init__(model_id=model_id, model=model, prompt=prompt, device=device)

    def prepare_texts(self, texts: List[str], **kwargs):
        """
        Prepares prompts for relation extraction to texts.

        Args:
            texts (list): List of input texts.

        Returns:
            list: List of formatted prompts.
        """
        prompts = []

        for id, text in enumerate(texts):
            prompt = f"{self.prompt} \n {text}"
            prompts.append(prompt)
        return prompts

    def prepare_source_relation(self, ner_predictions: List[dict], relations: List[str]):
        relation_labels = []
        for prediction in ner_predictions:
            curr_labels = []
            unique_entities = {ent['text'] for ent in prediction}
            for relation in relations:
                for ent in unique_entities:
                    curr_labels.append(f"{ent} <> {relation}")
            relation_labels.append(curr_labels)
        return relation_labels

    def process_predictions(self, predictions, **kwargs):
        """
        Processes predictions to extract relations, and if return_index=True,
        shifts any start/end by the exact number of characters prepended
        (prompt + " \\n ") so they align against the bare `text`.
        """
        batch_predicted_relations = []
        # account for prompt + space + newline + space
        shift = len(self.prompt) + len(" \n ")

        for prediction in predictions:
            curr_relations = []
            for target in prediction:
                source, rel_label = target['label'].split('<>')
                rel = {
                    "source":   source.strip(),
                    "relation": rel_label.strip(),
                    "target":   target['text'].strip(),
                    "score":    target['score']
                }

                if self.return_index:
                    raw_start = target.get('start')
                    raw_end   = target.get('end')
                    # subtract the exact prefix length
                    if raw_start is not None:
                        rel['start'] = raw_start - shift
                    if raw_end is not None:
                        rel['end']   = raw_end   - shift

                curr_relations.append(rel)

            batch_predicted_relations.append(curr_relations)

        return batch_predicted_relations

    def __call__(self, texts: Union[str, List[str]], relations: List[str]=None,
                                entities: List[str] = ['named entity'],
                                relation_labels: Optional[List[List[str]]]=None,
                                ner_threshold: float = 0.5,
                                rel_threshold: float = 0.5,
                                batch_size: int = 8, **kwargs):
        if isinstance(texts, str):
            texts = [texts]

        prompts = self.prepare_texts(texts, **kwargs)

        if relation_labels is None:
            # ner
            ner_predictions = self.model.run(texts, entities, threshold=ner_threshold, batch_size=batch_size)
            #rex
            relation_labels = self.prepare_source_relation(ner_predictions, relations)

        predictions = self.model.run(prompts, relation_labels, threshold=rel_threshold, batch_size=batch_size)

        results = self.process_predictions(predictions, **kwargs)

        return results

    def evaluate(self, dataset_id: Optional[str] = None, dataset: Optional[Dataset] = None,
                    labels: Optional[List[str]]=None, threshold: float =0.5, max_examples: float =-1):
        """
        Evaluates the model on a specified dataset and computes evaluation metrics.

        Args:
            dataset_id (str, optional): Identifier for the dataset to load (e.g., from Hugging Face datasets).
            dataset (Dataset, optional): A pre-loaded dataset to evaluate. If provided, `dataset_id` is ignored.
            labels (list, optional): List of target labels to consider for relation extraction. Defaults to None (use all).
            threshold (float): Confidence threshold for predictions. Defaults to 0.5.
            max_examples (int): Maximum number of examples to evaluate. Defaults to -1 (use all available examples).

        Returns:
            dict: A dictionary containing evaluation metrics such as F1 scores.

        Raises:
            ValueError: If neither `dataset_id` nor `dataset` is provided.
        """
        raise NotImplementedError("Currently `evaluate` method is not implemented.")

In [ ]:
# your imports
from tqdm.auto import tqdm
from typing import List, Dict, Any, Tuple

# your globals


labels = ['named dataset', 'unnamed dataset', 'vague dataset']
rels = ['acronym', 'author', 'data description', 'data geography', \
       'reference year','data type', 'publication year', 'publisher', \
        'reference population', 'version', 'data source']

TYPE2RELS = {
    "named dataset":   rels,
    "unnamed dataset": rels,
    "vague dataset":   rels,
}

#
def inference_pipeline(
    text: str,
    model,
    labels: List[str],
    relation_extractor_custom: CustomGLiNERRelationExtractor,
    TYPE2RELS: Dict[str, List[str]],
    ner_threshold: float = 0.5,
    rel_threshold: float = 0.5,
    re_multi_label: bool = False,
    return_index: bool = False,
) -> Tuple[List[Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    ner_preds = model.predict_entities(
        text,
        labels,
        flat_ner=False,
        threshold=ner_threshold
    )

    re_results: Dict[str, List[Dict[str, Any]]] = {}
    for ner in ner_preds:
        span       = ner['text']
        rel_types  = TYPE2RELS.get(ner['label'], [])
        if not rel_types:
            continue

        slot_labels = [f"{span} <> {r}" for r in rel_types]

        preds = relation_extractor_custom(
            text,
            relations=None,
            entities=None,
            relation_labels=slot_labels,
            threshold=rel_threshold,
            multi_label=re_multi_label,
            return_index=return_index,
            distance_threshold=25,
        )[0]

        re_results[span] = preds

    return ner_preds, re_results

In [ ]:
import pandas as pd

# Metric functions with empty‐empty as perfect
def get_precision_recall(tp, fp, fn):
    """
    If there are no positives anywhere (tp+fp==0 and tp+fn==0),
    treat as perfect. Otherwise use the usual definitions.
    """
    if (tp + fp) == 0 and (tp + fn) == 0:
        return 1.0, 1.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall

def fbeta_score(precision, recall, beta=1):
    """
    Standard F-beta, with guard against zero denominator.
    """
    denom = (beta*beta*precision) + recall
    return ((1 + beta*beta) * precision * recall / denom) if denom > 0 else 0.0

def jaccard(str1, str2):
    """
    Jaccard similarity on token sets, returns 0 if both empty.
    """
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a & b
    denom = len(a) + len(b) - len(c)
    return len(c) / denom if denom > 0 else 0.0

def coleridge_initiative_jaccard(ground_truth: str, prediction: str, verbose=True):
    """
    Compute TP/FP/FN via >=0.5 Jaccard threshold on |-separated "label: text" strings.
    """
    gts = [g for g in ground_truth.split("|") if g.strip()]
    pds = [p for p in prediction.split("|")    if p.strip()]
    if verbose:
        print("GT:", gts)
        print("PD:", pds)

    js_scores = []
    cf = []

    # no predictions => all FNs
    if not pds:
        for gt in gts:
            js_scores.append(0.0); cf.append("FN")
        return js_scores, " ".join(cf)
    # no ground truth => all FPs
    if not gts:
        for pd in pds:
            js_scores.append(0.0); cf.append("FP")
        return js_scores, " ".join(cf)

    # TP/FP
    for pd in pds:
        best = max(jaccard(pd, gt) for gt in gts)
        if best >= 0.5:
            js_scores.append(best); cf.append("TP")
        else:
            js_scores.append(best); cf.append("FP")
    # FN
    for gt in gts:
        best = max(jaccard(gt, pd) for pd in pds)
        if best < 0.5:
            js_scores.append(best); cf.append("FN")

    return js_scores, " ".join(cf)

def evaluate_metrics(ground_truth: list[str], predictions: list[str]):
    """
    Runs the Jaccard‐based TP/FP/FN over all examples, then computes
    overall precision, recall, and F1.
    """
    overall_tp = overall_fp = overall_fn = 0
    for gt, pred in zip(ground_truth, predictions):
        _, cf_str = coleridge_initiative_jaccard(gt, pred, verbose=False)
        counts = cf_str.split()
        overall_tp += counts.count("TP")
        overall_fp += counts.count("FP")
        overall_fn += counts.count("FN")

    precision, recall = get_precision_recall(overall_tp, overall_fp, overall_fn)
    f1 = fbeta_score(precision, recall, beta=1)
    return {
        "tp": overall_tp,
        "fp": overall_fp,
        "fn": overall_fn,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

In [ ]:
def sample_confusion(gt_str, pred_str, threshold=0.5):
    """
    Given two '|'‐joined strings of "label: text", returns
      - tp: list of preds counted as true positives
      - fp: list of preds counted as false positives
      - fn: list of gts counted as false negatives
    """
    gts = [g for g in gt_str.split("|")   if g.strip()]
    pds = [p for p in pred_str.split("|") if p.strip()]

    fp = []
    tp = []
    # classify preds
    for pd in pds:
        if not gts:
            fp.append(pd)
            continue
        best = max(jaccard(pd, gt) for gt in gts)
        if best >= threshold:
            tp.append(pd)
        else:
            fp.append(pd)
    # classify ground truths
    fn = []
    for gt in gts:
        if not pds:
            fn.append(gt)
            continue
        best = max(jaccard(gt, pd) for pd in pds)
        if best < threshold:
            fn.append(gt)
    return tp, fp, fn

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
TARGET_LABELS = {"named dataset", "unnamed dataset", "vague dataset"}

def to_metric_string_unique(label_text_list: list[tuple[str,str]]) -> str:
    """Same as to_metric_string but dedupe by text only."""
    seen = set()
    pieces = []
    for lbl, txt in label_text_list:
        if txt in seen:
            continue
        seen.add(txt)
        if lbl in TARGET_LABELS:
            pieces.append(f"{lbl}: {txt}")
    return "|".join(pieces)

def to_metric_string(label_text_list):
    """
    label_text_list: e.g. [('named dataset','Survey'),(...)]
    Keep only the TARGET_LABELS, format as 'label: text', join with '|'.
    """
    pieces = [f"{lbl}: {txt}" for lbl, txt in label_text_list if lbl in TARGET_LABELS]
    return "|".join(pieces)

def prune_self_and_short_acronyms(ner_preds, rel_preds):
    """
    Removes:
      • self‐relations (where target == source)
      • acronym relations where len(source) < len(target)
    Leaves all other entities and relations untouched.
    """
    # keep all entities
    filtered_ner = ner_preds.copy()

    # filter relations
    filtered_re = {}
    for src, rels in rel_preds.items():
        kept = []
        for r in rels:
            # drop self‐relations
            if r['target'] == src:
                continue
            # # drop acronym where the abbrev is shorter than the full name
            # if r['relation'] == 'acronym' and len(src) < len(r['target']):
            #     continue
            kept.append(r)
        if kept:
            filtered_re[src] = kept

    return filtered_ner, filtered_re

def evaluate_model_on_holdout(
    model,
    holdout_df: pd.DataFrame,
    inference_fn,
    ner_threshold=0.5,
    rel_threshold=0.5,
    prune_acronyms=False,
    use_gr=True,
):
    """
    Runs your inference pipeline over holdout_df, computes row‐level TP/FP/FN
    and then overall precision/recall/F1.
    Returns (metrics_dict, detailed_df).
    """
    records = []
    relation_extractor_custom = CustomGLiNERRelationExtractor(model=model, return_index=True)

    for _, row in tqdm(holdout_df.iterrows(), total=len(holdout_df), desc="Holdout inference"):
        text = row["text"]
        ner_preds, rel_preds = inference_fn(
            text,
            model=model,
            labels=labels,
            relation_extractor_custom=relation_extractor_custom,
            TYPE2RELS=TYPE2RELS,
            ner_threshold=ner_threshold,
            rel_threshold=rel_threshold,
            re_multi_label=False,
            return_index=True,
        )
        if prune_acronyms:
          ner_preds, rel_preds = prune_acronym_and_self_relations(ner_preds, rel_preds)
        else:
          ner_preds, rel_preds = prune_self_and_short_acronyms(ner_preds, rel_preds)
        if use_gr:
          gr_row = row["ground_truth_reviewed"]
        else:
          gr_row = row["predictions_validated"]
        records.append({
            "filename": row["filename"],
            "ground_truth_reviewed": gr_row,
            "predictions": ner_preds
        })
    df = pd.DataFrame(records)
    # build the strings
    df['gt_str_u'] = df['ground_truth_reviewed'].apply(
        lambda ents: to_metric_string([(e['label'], e['text']) for e in ents])
    )
    df['pred_label_text'] = df['predictions'] \
        .apply(lambda preds: [(p['label'], p['text']) for p in preds])

    df['pred_str_u'] = df['pred_label_text'].apply(to_metric_string)

    df[["TP","FP","FN"]] = df.apply(
        lambda r: pd.Series(sample_confusion(r["gt_str_u"], r["pred_str_u"])),
        axis=1
    )
    # overall
    metrics = evaluate_metrics(df["gt_str_u"].tolist(), df["pred_str_u"].tolist())
    return metrics, df

In [ ]:
def combine_preds_and_rels(row):
    out = []
    # add all predictions
    for p in row["predictions"]:
        out.append({
            "label": p["label"],
            "text":  p["text"],
            "start": p["start"],
            "end":   p["end"]
        })
    # flatten your relations dict into the same format
    for rel_list in row["relations"].values():
        for r in rel_list:
            out.append({
                "label": r["relation"],
                "text":  r["target"],
                "start": r["start"],
                "end":   r["end"]
            })
    # sort by start so they're in document order
    out.sort(key=lambda x: x["start"])
    return out

# Define the Holdout Data

In [ ]:
from collections import defaultdict

def group_relations(rels):
    grouped = defaultdict(list)
    for r in rels:
        grouped[r['head']].append({
            'source': r['head'],
            'relation': r['label'],
            'target': r['tail'],
            'start': r['start'],
            'end': r['end']
        })
    return dict(grouped)

valid_labels = {'named dataset', 'unnamed dataset', 'vague dataset'}

def filter_predictions(preds):
    return [p for p in preds if p.get('label') in valid_labels]

In [ ]:
import os, json
from glob import glob
from typing import List
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def load_synthetic_results_parallel(root_dir: str, max_workers: int = 8) -> List[dict]:
    """
    Walks through all subfolders under `root_dir` and loads every JSON file
    in parallel, returning a flat list of dicts.
    """
    # find all json files under any immediate subfolder
    pattern = os.path.join(root_dir, '*', '*.json')
    paths = glob(pattern)

    results = []
    def _load(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        # kick off all loads
        futures = {exe.submit(_load, p): p for p in paths}
        # as each one completes, collect or warn
        for fut in tqdm(as_completed(futures), total=len(futures), desc="loading synthetic"):
            path = futures[fut]
            try:
                results.append(fut.result())
            except Exception as e:
                print(f"⚠️ failed to load {path}: {e}")

    return results

import os
import json
from glob import glob
from typing import Any, List
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def load_and_flatten_revalidated_parallel(
    root_dir: str,
    max_workers: int = 8
) -> List[Any]:
    """
    Loads every JSON file in *root_dir* (and its subfolders) in parallel,
    each file containing one revalidated entry, and returns one flat list of all entries.
    """
    # find all json files under root_dir (recursively)
    pattern = os.path.join(root_dir, '**', '*.json')
    paths = glob(pattern, recursive=True)

    results: List[Any] = []

    def _load(path: str) -> Any:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        # submit all loads
        future_to_path = {exe.submit(_load, p): p for p in paths}
        for fut in tqdm(as_completed(future_to_path),
                        total=len(future_to_path),
                        desc="loading revalidated data"):
            path = future_to_path[fut]
            try:
                data = fut.result()
                if isinstance(data, dict):
                    results.append(data)
                else:
                    print(f"{path} did not yield a dict, skipping.")
            except Exception as e:
                print(f"failed to load {path}: {e}")

    return results

In [ ]:
import pandas as pd
import os
import ast
holdout_data = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/holdout_data_manual_reviewed.csv"
holdout_data = pd.read_csv(holdout_data)
holdout_data['ground_truth_reviewed'] = holdout_data['ground_truth_reviewed'].apply(ast.literal_eval)
# define used documents to not leak data, and also for looping in active learning


In [ ]:
def detokenize(tokens):
    """
    Naïvely glue tokens back into a string (you can swap this out
    for whatever spacing logic you need).
    """
    # insert a space before each token except punctuation
    out = []
    for t in tokens:
        if out and not re.match(r"^[,.;:!?)]$", t):
            out.append(" ")
        out.append(t)
    return "".join(out)

def build_doc_from_token_level(row):
    """
    row: {
      'tokenized_text': List[str],
      'ner':  List of [start_token_idx, end_token_idx, label],
      're':   List of [h_s, h_e, t_s, t_e, relation_label]
    }
    """
    tokens = row["tokenized_text"]
    text   = detokenize(tokens)

    # build predictions
    predictions = []
    for s_tok, e_tok, lbl in row["ner"]:
        # grab the text of those tokens
        span_txt = detokenize(tokens[s_tok : e_tok+1])
        # find the character offsets in the detokenized string
        char_start = text.find(span_txt)
        char_end   = char_start + len(span_txt)
        predictions.append({
            "start": char_start,
            "end":   char_end,
            "text":  span_txt,
            "label": lbl
        })

    # build relations: group by head text
    relations = {}
    for h_s, h_e, t_s, t_e, rel_lbl in row["re"]:
        head_txt  = detokenize(tokens[h_s : h_e+1])
        targ_txt  = detokenize(tokens[t_s : t_e+1])
        # char spans for tail
        t0 = text.find(targ_txt)
        t1 = t0 + len(targ_txt)
        rel = {
            "source":   head_txt,
            "relation": rel_lbl,
            "target":   targ_txt,
            "start":    t0,
            "end":      t1
        }
        relations.setdefault(head_txt, []).append(rel)

    return {
        "text":        text,
        "filename": "initial",
        "page": 9999,
        "predictions": predictions,
        "relations":   relations
    }

In [ ]:
def map_indices_to_substring(text, start, end):
    """
    Function to map the start and end indices to the corresponding substring in the provided text.

    Parameters:
    text (str): The text from which the substring is to be extracted.
    start (int): The starting index of the substring.
    end (int): The ending index of the substring.

    Returns:
    str: The substring from the provided text based on the start and end indices.
    """
    return text[start:end]

# Revalidate Initial data & Generate Synthetic data

In [ ]:
# get client
from openai import OpenAI
from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
# Load API keys from secret manager
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Initialize OpenAI client
client = OpenAI()
OPENAI_MODEL = 'gpt-4o-mini'  # or 'gpt-4o-mini-2024-07-18'
SYNTH_MODEL = 'gpt-4o-mini'

## After revalidation and generation process the outputs

In [ ]:
model = load_model('knowledgator/gliner-multitask-v1.0')

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

spm.model:   0%|          | 0.00/2.45M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.64M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

gliner_config.json:   0%|          | 0.00/3.79k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

In [ ]:
# now evaluate on holdout
history = []
detailed_df_all = pd.DataFrame()

metrics, detail_df = evaluate_model_on_holdout(
    model=model,
    holdout_df=holdout_data,
    inference_fn=inference_pipeline, # inference pipeline
    ner_threshold=0.5,
    rel_threshold=0.5
)
history.append({
    "round": 0,
    **metrics
})

detail_df['round'] = 0

detailed_df_all = pd.concat([detailed_df_all, detail_df], axis=0, ignore_index=True)

Holdout inference:   0%|          | 0/110 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
pd.DataFrame(history)

,round,tp,fp,fn,precision,recall,f1_score
0,0,8,0,241,1.0,0.032129,0.062257


# Initiate Active Learning Loop

In [ ]:
import random
HOLDOUT_DOCUMENTS = {f"{fn}.json" for fn in holdout_data['filename'].unique()}
TRAIN_DATA_INITIAL_FNAMES = ['A-Reappraisal-of-the-Migration-Development-Nexus-Testing-the-Robustness-of-the-Migration-Transition-Hypothesis',
 'Does-School-Safety-and-Classroom-Disciplinary-Climate-Hinder-Learning-Evidence-from-the-MENA-Region',
 'Economic-and-Fiscal-Impacts-of-Venezuelan-Refugees-and-Migrants-in-Brazil',
 'Big-Data-for-Sampling-Design-The-Venezuelan-Migration-Crisis-in-Ecuador',
 'Estimating-Poverty-among-Refugee-Populations-A-Cross-Survey-Imputation-Exercise-for-Chad',
 'Forced-Migration-Social-Cohesion-and-Conflict-The-2015-Refugee-Inflow-in-Germany',
 'Growth-after-War-in-Syria',
 'How-Do-Gender-Norms-Shape-Education-and-Domestic-Work-Outcomes-The-Case-of-Syrian-Refugee-Adolescents-in-Jordan',
 'How-to-Cope-with-a-Refugee-Shock-Evidence-from-Uganda',
 'Immigration-Labor-Markets-and-Discrimination-Evidence-from-the-Venezuelan-Exodus-in-Peru',
 'Children-on-the-Move-Progressive-Redistribution-of-Humanitarian-Cash-Transfers-among-Refugees',
 'Inclusive-Refugee-Hosting-in-Uganda-Improves-Local-Development-and-Prevents-Public-Backlash',
 'Is-Informality-Good-for-Business-The-Impacts-of-IDP-Inflows-on-Formal-Firms',
 'Liberalizing-versus-Facilitating-Mode-4-Trade-in-Services',
 'Life-out-of-the-Shadows-The-Impacts-of-Regularization-Programs-on-the-Lives-of-Forced-Migrants',
 'Local-Peace-Agreements-and-the-Return-of-IDPs-with-Perceived-ISIL-Affiliation-in-Iraq',
 'Long-Term-Effects-of-the-1923-Mass-Refugee-Inflow-on-Social-Cohesion-in-Greece',
 'Rohingya-Refugee-Camps-and-Forest-Loss-in-Cox-s-Bazar-Bangladesh-An-Inquiry-Using-Remote-Sensing-and-Econometric-Approaches',
 'Sharing-Responsibility-through-Joint-Decision-Making-and-Implications-for-Intimate-Partner-Violence-Evidence-from-12-Sub-Saharan-African-Countries',
 'Shoring-Up-Economic-Refugees-Venezuelan-Migrants-in-the-Ecuadoran-Labor-Market',
 'Social-Cohesion-and-Refugee-Host-Interactions-Evidence-from-East-Africa',
 'The-Geography-of-Displacement-Refugees-Camps-and-Social-Conflicts',
 'Coping-with-the-Influx-Service-Delivery-to-Syrian-Refugees-and-Hosts-in-Jordan-Lebanon-and-Kurdistan-Iraq',
 'The-Globalization-of-Refugee-Flows',
 'The-Impact-of-Living-Arrangements-In-Camp-versus-Out-of-Camp-on-the-Quality-of-Life-A-Case-Study-of-Syrian-Refugees-in-Jordan',
 'The-Lives-and-Livelihoods-of-Syrian-Refugees-in-the-Middle-East-Evidence-from-the-2015-16-Surveys-of-Syrian-Refugees-and-Host-Communities-in-Jordan-Lebanon-and-Kurdistan-Iraq',
 'The-Risk-That-Travels-with-You-Links-between-Forced-Displacement-Conflict-and-Intimate-Partner-Violence-in-Colombia-and-Liberia',
 'The-Syrian-Refugee-Life-Study-First-Glance',
 'The-Unintended-Consequences-of-Deportations-Evidence-from-Firm-Behavior-in-El-Salvador',
 'Updating-poverty-estimates-at-frequent-intervals-in-the-absence-of-consumption-data-methods-and-illustration-with-reference-to-a-middle-income-country',
 'Differences-in-Household-Composition-Hidden-Dimensions-of-Poverty-and-Displacement-in-Somalia']
TRAIN_DATA_INITIAL_FNAMES = {f"{fn}.json" for fn in TRAIN_DATA_INITIAL_FNAMES}
ALL_DOCUMENTS  = set(os.listdir(
    '/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/document_sentences/'
))

candidates = list(ALL_DOCUMENTS - HOLDOUT_DOCUMENTS - TRAIN_DATA_INITIAL_FNAMES)
candidates = [c for c in candidates if '.json' in c]
random.shuffle(candidates)

# Pre-slice into batches of 10
batch_size = 10
batches = [
    candidates[i : i + batch_size]
    for i in range(0, len(candidates), batch_size)
]

In [ ]:
len(HOLDOUT_DOCUMENTS), len(TRAIN_DATA_INITIAL_FNAMES), len(ALL_DOCUMENTS)

(17, 31, 133)

In [ ]:
len(candidates), len(batches)

(84, 9)

In [ ]:
sentences_path = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/document_sentences/"

In [ ]:
from typing import List, Dict, Any

TARGET_LABELS = {"named dataset", "unnamed dataset", "vague dataset"}
import math

def select_top_uncertain_mentions(
    docs: List[Dict[str, Any]],
    top_k: int
) -> List[Dict[str, Any]]:
    """
    From a list of inference-output docs, pick the top_k most uncertain
    dataset mentions stratified 50% named / 25% unnamed / 25% vague,
    then build mini-docs each containing only that one prediction
    (and its matching relations) for targeted revalidation.

    If a bucket has fewer than its quota, we backfill from the remaining
    mentions sorted by uncertainty.

    Args:
        docs: each dict with keys 'filename','page','text','predictions','relations'
        top_k: total number of mentions to return

    Returns:
        List of dicts, each with exactly one 'predictions' entry and
        only those relations whose source matches that prediction's text.
    """
    # flatten into mention-level candidates
    mention_cands = []
    for d in docs:
        for p in d.get("predictions", []):
            lbl = p.get("label")
            if lbl not in TARGET_LABELS:
                continue
            unc = 1.0 - p.get("score", 0.0)
            mention_cands.append({
                "doc":         d,
                "prediction":  p,
                "uncertainty": unc
            })

    # sort descending by uncertainty
    mention_cands.sort(key=lambda m: m["uncertainty"], reverse=True)

    # allocate quotas
    n_named   = math.floor(top_k * 0.70)
    n_unnamed = math.floor(top_k * 0.20)
    n_vague   = top_k - n_named - n_unnamed

    # bucket by label
    buckets = {"named dataset": [], "unnamed dataset": [], "vague dataset": []}
    for m in mention_cands:
        lbl = m["prediction"]["label"]
        buckets[lbl].append(m)

    selected = []
    # take from each bucket up to its quota
    for lbl, quota in (("named dataset", n_named),
                       ("unnamed dataset", n_unnamed),
                       ("vague dataset", n_vague)):
        selected.extend(buckets[lbl][:quota])

    # back-fill if we didn't hit top_k
    if len(selected) < top_k:
        # build a set of already chosen mention ids
        chosen = { id(m["prediction"]) for m in selected }
        for m in mention_cands:
            if len(selected) >= top_k:
                break
            if id(m["prediction"]) in chosen:
                continue
            selected.append(m)

    # reconstruct one-prediction docs
    docs_for_reval = []
    for m in selected:
        d = m["doc"]
        p = m["prediction"]
        pr = [p]
        # only relations whose source exactly matches this text
        rels = {
            src: rel_list
            for src, rel_list in d.get("relations", {}).items()
            if src == p["text"]
        }
        docs_for_reval.append({
            "filename":    d["filename"],
            "page":        d["page"],
            "text":        d["text"],
            "predictions": pr,
            "relations":   rels
        })

    return docs_for_reval

def infer_new_batch(all_doc_this_batch, model):
  rel_ex = CustomGLiNERRelationExtractor(model=model, return_index=True)
  for item in tqdm(all_doc_this_batch, desc="infer on new batch"):
      text = item.get("text", "")
      ner_preds, rel_preds = inference_pipeline(
          text,
          model=model,
          labels=labels,
          relation_extractor_custom=rel_ex,
          TYPE2RELS=TYPE2RELS,
          ner_threshold=0.5,
          rel_threshold=0.5,
          re_multi_label=False,
          return_index=True,
      )
      # Add the predictions directly into each dict
      ner_preds, rel_preds = prune_self_and_short_acronyms(ner_preds, rel_preds)
      item["predictions"] = ner_preds
      item["relations"]   = rel_preds

  return all_doc_this_batch

def get_subsample_frac(
    epoch: int,
    start_frac: float = 1.0,
    decay: float  = 0.05,
    min_frac: float= 0.0
) -> float:
    """
    Compute a subsample fraction that starts at `start_frac`
    and decreases by `decay` each epoch, floored at `min_frac`.

    Args:
        epoch:     1-based training round
        start_frac: initial fraction at epoch=1 (e.g. 1.0 = 100%)
        decay:     fraction to subtract per epoch (e.g. 0.05 = −5%)
        min_frac:  lower bound (e.g. 0.1 = 10%)
    """
    frac = start_frac - decay * (epoch - 1)
    return max(frac, min_frac)

In [ ]:
def get_accumulating_frac(
    epoch: int,
    start_frac: float = 0.5,
    growth: float     = 0.05,
    max_frac: float   = 1.0
) -> float:
    """
    Compute a subsample fraction that starts at `start_frac`
    and increases by `growth` each epoch, capped at `max_frac`.

    Args:
        epoch:      1-based training round
        start_frac: initial fraction at epoch=1 (e.g. 0.0 = 0%)
        growth:     fraction to add per epoch (e.g. 0.05 = +5%)
        max_frac:   upper bound (e.g. 1.0 = 100%)
    """
    frac = start_frac + growth * (epoch - 1)
    return min(frac, max_frac)


In [ ]:
def train_gliner_model_continual(
    model,
    synthetic_dataset: list,
    validated_dataset: list,
    subsample_frac: float = 0.5,
    num_steps: int = 500,
    base_batch_size: int = 4,
):
    """
    Continual finetuning: replay a small subsample of your original
    HQ + synthetic “initial” data, then train on that + the newly
    generated synthetic & validated entries.

    Args:
      model: a GLiNER model (already loaded/pretrained).
      synthetic_dataset: newly generated synthetic examples (list of dicts).
      validated_dataset: newly revalidated HQ examples (list of dicts).
      initial_train: if True, use the full initial dumps (not just subsample).
      subsample_frac: fraction of initial dumps to replay each round.
      num_steps: approximate total optimization steps.
      base_batch_size: per-device train/eval batch size.
    Returns:
      The same model, finetuned in place.
    """
    # load dumps
    init_hq_path    = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/train_data_initial.json"
    init_syn_path   = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/synthetic_data_initial.json"
    initial_hq      = load_json_data(init_hq_path)    # list of dicts
    initial_synth   = load_json_data(init_syn_path)   # list of dicts

    model = load_model('knowledgator/gliner-multitask-v1.0')

    n_hq    = max(1, int(len(initial_hq)  * subsample_frac))
    n_syn   = max(1, int(len(initial_synth)* subsample_frac))
    replay_hq    = random.sample(initial_hq,    n_hq)
    replay_synth = random.sample(initial_synth, n_syn)

    # build the overall train pool
    train_pool_hq = replay_hq + validated_dataset
    train_pool_synth = replay_synth + synthetic_dataset

    print(f"length of entire train pool: {len(train_pool_hq)}")
    print(f"length of synthetic pool: {len(train_pool_synth)}")

    # split into train / eval
    train_set_synth, eval_set_synth = make_train_test_split(train_pool_synth)
    train_set_hq, eval_set_hq = make_train_test_split(train_pool_hq)


    # prepare trainer
    data_collator = DataCollator(
        model.config,
        data_processor=model.data_processor,
        prepare_labels=True
    )
    # derive number of epochs from num_steps
    num_batches = max(1, len(train_set_synth) // base_batch_size)
    num_epochs = max(1, num_steps // num_batches)

    training_args = TrainingArguments(
        output_dir="models/continual",
        learning_rate=1e-5,
        weight_decay=0.01,
        others_lr=1e-5,
        others_weight_decay=0.01,
        lr_scheduler_type="cosine",   # for finetuning
        warmup_ratio=0.1,
        per_device_train_batch_size=base_batch_size,
        per_device_eval_batch_size=base_batch_size,
        focal_loss_alpha=0.75,
        focal_loss_gamma=2,
        num_train_epochs=num_epochs,
        eval_strategy="steps",
        save_steps=250,
        save_total_limit=10,
        dataloader_num_workers=0,
        use_cpu=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_set_synth,
        eval_dataset=eval_set_synth,
        data_collator=data_collator,
    )

    # prefinetune
    trainer.train()


    # finetune
    # derive number of epochs from num_steps
    num_batches = max(1, len(train_set_hq) // base_batch_size)
    num_epochs = max(1, num_steps // num_batches)

    training_args = TrainingArguments(
        output_dir="models/continual",
        learning_rate=5e-6,
        weight_decay=0.01,
        others_lr=1e-5,
        others_weight_decay=0.01,
        lr_scheduler_type="linear",   # for finetuning
        warmup_ratio=0.1,
        per_device_train_batch_size=base_batch_size,
        per_device_eval_batch_size=base_batch_size,
        focal_loss_alpha=0.75,
        focal_loss_gamma=2,
        num_train_epochs=num_epochs,
        eval_strategy="steps",
        save_steps=250,
        save_total_limit=5,
        dataloader_num_workers=0,
        use_cpu=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_set_hq,
        eval_dataset=eval_set_hq,
        data_collator=data_collator,
    )

    # 6) actually finetune
    trainer.train()


    return model

In [ ]:
import re

def is_noise_text(text: str,
                  min_alpha_ratio: float = 0.5,
                  max_uppercase_word_ratio: float = 0.6,
                  min_word_count: int = 5) -> bool:
    """
    Heuristic filter for “noisy” texts that look like table dumps or
    OCR‐style garbage:
      1) Too few real words
      2) Not enough alphabetic characters overall
      3) Too many ALL‐CAP words (often headings / noise)
    """
    words = text.split()
    if len(words) < min_word_count:
        return True

    # alphabetic ratio
    alpha_chars = sum(c.isalpha() for c in text)
    if alpha_chars / max(len(text), 1) < min_alpha_ratio:
        return True

    # uppercase word ratio
    uppercase_words = sum(1 for w in words if w.isupper() and re.search(r"[A-Z]", w))
    if uppercase_words / len(words) > max_uppercase_word_ratio:
        return True

    return False

In [ ]:
from copy import deepcopy

def filter_revalidated(docs, keep_actions=('keep','update')):
    """
    For each revalidated doc, keep only those predictions & relations
    whose action is in keep_actions, *and* drop any relation whose source
    no longer appears in the kept predictions.
    """
    filtered = []
    for doc in docs:
        new_doc = deepcopy(doc)
        new_doc.pop('ner_text_validated', None)
        # keep only the desired predictions
        new_preds = [
            p for p in doc.get('predictions_validated', [])
            if p.get('action') in keep_actions
        ]
        kept_texts = {p['text'] for p in new_preds}

        # keep only relations with allowed action and source in kept_texts
        new_rels = {}
        for src, rels in doc.get('relations_validated', {}).items():
            if src not in kept_texts:
                continue
            kept = [r for r in rels if r.get('action') in keep_actions]
            if kept:
                new_rels[src] = kept

        # only include this doc if there's at least one surviving prediction
        if new_preds:
            new_doc['predictions_validated'] = new_preds
            new_doc['relations_validated']  = new_rels
            filtered.append(new_doc)

    return filtered

In [ ]:
def build_validated_ner_char(preds, rels):
    """
    Merge validated predictions and relations into character-level NER format.
    Returns list of [start, end, payload_string].
    """
    payloads = []
    # Add each kept prediction with label only
    for p in preds:
        payloads.append([p['start'], p['end'], p['label']])
    # Add each kept relation using its own span, label as source <> relation
    for src, rel_list in rels.items():
        for r in rel_list:
            payloads.append([r['start'], r['end'], f"{r['source']} <> {r['relation']}"])
    # Sort by start then end
    payloads.sort(key=lambda x: (x[0], x[1]))
    return payloads

In [ ]:
def post_process_revalidated(revalidated_results):
  # then filter your revalidated_results:
  rev_results = [
      doc for doc in revalidated_results
      if not is_noise_text(doc["text"])
  ]
  rev_results_filtered = filter_revalidated(rev_results)
  # re‐apply build_validated_ner_char to every document:
  for doc in rev_results_filtered:
      preds = doc.get("predictions_validated", [])
      rels  = doc.get("relations_validated", {})
      doc["ner_text_validated"] = build_validated_ner_char(preds, rels)
  return rev_results_filtered

In [ ]:
import math
from typing import List, Dict, Any

def select_top_uncertain_and_certain_mentions(
    docs: List[Dict[str, Any]],
    top_k: int
) -> List[Dict[str, Any]]:
    """
    Selects top_k dataset mentions for revalidation, stratified by label
    (named/unnamed/vague) and by score (half most uncertain, half most certain).

    Args:
        docs: List of docs with 'predictions' and 'relations'
        top_k: Number of mentions to return

    Returns:
        List of mini-docs, each with 1 prediction and its matching relations.
    """
    # Flatten into mention-level candidates
    mention_cands = []
    for d in docs:
        for p in d.get("predictions", []):
            lbl = p.get("label")
            if lbl not in {"named dataset", "unnamed dataset", "vague dataset"}:
                continue
            score = p.get("score", 0.0)
            mention_cands.append({
                "doc":        d,
                "prediction": p,
                "score":      score
            })

    # Quota by label
    n_named   = math.floor(top_k * 0.7)
    n_unnamed = math.floor(top_k * 0.15)
    n_vague   = top_k - n_named - n_unnamed

    quotas = {
        "named dataset": n_named,
        "unnamed dataset": n_unnamed,
        "vague dataset": n_vague
    }

    selected = []

    for label, quota in quotas.items():
        bucket = [m for m in mention_cands if m["prediction"]["label"] == label]
        if not bucket or quota == 0:
            continue
        # Sort by score: ascending (uncertain) and descending (certain)
        low_conf  = sorted(bucket, key=lambda m: m["score"])         # most uncertain
        high_conf = sorted(bucket, key=lambda m: m["score"], reverse=True)  # most certain

        n_low  = quota // 2
        n_high = quota - n_low

        # Take the quotas (handle small buckets gracefully)
        picked_low  = low_conf[:n_low]
        picked_high = [m for m in high_conf if m not in picked_low][:n_high]

        selected.extend(picked_low + picked_high)

    # Back-fill if we didn't hit top_k
    if len(selected) < top_k:
        chosen = {id(m["prediction"]) for m in selected}
        # Take the rest sorted by *uncertainty* (lowest scores)
        rest = sorted(
            [m for m in mention_cands if id(m["prediction"]) not in chosen],
            key=lambda m: m["score"]
        )
        selected.extend(rest[:(top_k - len(selected))])

    # 3) Reconstruct one-prediction docs
    docs_for_reval = []
    for m in selected:
        d = m["doc"]
        p = m["prediction"]
        pr = [p]
        rels = {
            src: rel_list
            for src, rel_list in d.get("relations", {}).items()
            if src == p["text"]
        }
        docs_for_reval.append({
            "filename":    d["filename"],
            "page":        d["page"],
            "text":        d["text"],
            "predictions": pr,
            "relations":   rels
        })
    return docs_for_reval


In [ ]:
model = load_model("rafmacalaba/gliner-re-init-model")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

gliner_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.64M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.45M [00:00<?, ?B/s]

In [ ]:
!rm -r /content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/revalidated_res_continual/
!rm -r /content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/synthetic_continual/

In [ ]:
best_f1 = metrics['f1_score']
no_improve = 0
patience = 3
# Loop over batches
for epoch, batch in enumerate(batches, start=1):
    print(f"\n=== Epoch {epoch}: processing batch of {len(batch)} docs ===")
    all_doc_this_batch = []
    for doc in tqdm(batch, desc="loading batches"):
      all_doc_this_batch.extend(load_json_data(sentences_path + doc))
    all_doc_this_batch = [
        doc for doc in all_doc_this_batch
        if not is_noise_text(doc["text"])
    ]
    all_doc_this_batch = infer_new_batch(all_doc_this_batch, model)
    all_doc_this_batch = [doc for doc in all_doc_this_batch if doc['predictions']]

    docs_for_reval = select_top_uncertain_and_certain_mentions(all_doc_this_batch, top_k=100)

    rev_results_this_batch = revalidate_and_backup(
        docs=docs_for_reval,
        llm_revalidate_fn=llm_revalidate_with_relations,
        build_ner_fn=build_validated_ner_char,
        backup_dir="/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/revalidated_res_continual/"
    )
    base_path_synth = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/synthetic_continual/"
    synthetic_res_this_batch = generate_and_backup_synthetic(rev_results_this_batch, generate_synthetic_data, base_path_synth, n=1)

    synthetic_root = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/synthetic_continual/"
    synthetic_results = load_synthetic_results_parallel(synthetic_root, max_workers=12)

    reval_root = "/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/revalidated_res_continual/"
    revalidated_results = load_and_flatten_revalidated_parallel(reval_root, max_workers=12)
    revalidated_results = post_process_revalidated(revalidated_results)
    rev_results_processed = process_revalidated_data(revalidated_results)
    synthetic_results_processed = process_synthetic_data(synthetic_results)

    print(f'''Total new revalidated entries loaded: {len(rev_results_processed)} \n
    Total new synthetic entries loaded: {len(synthetic_results_processed)}''')

    subsample_frac = get_accumulating_frac(epoch, start_frac=0.5, growth=0.1, max_frac=1.0)
    print(f"Epoch {epoch}: replaying {subsample_frac*100:.1f}% of the initial data")

    model = train_gliner_model_continual(
        model,
        synthetic_dataset=synthetic_results_processed,
        validated_dataset=rev_results_processed,
        subsample_frac=subsample_frac
    )


    print(f"training model on epoch {epoch}")
    metrics, detail_df = evaluate_model_on_holdout(
        model=model,
        holdout_df=holdout_data,      # text & ground_truth_reviewed
        inference_fn=inference_pipeline, # inference pipeline
        ner_threshold=0.5,
        rel_threshold=0.5
    )

    detail_df["round"] = epoch
    detailed_df_all = pd.concat([detailed_df_all, detail_df], ignore_index=True)

    # record history
    history.append({
        "round": epoch,
        **metrics
    })
    history_df = pd.DataFrame(history)

    print(f'''holdout metrics = {metrics}
    holdout F1 = {metrics['f1_score']:.4f}''')
    ckpt_path_epoch = f"/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/models/round_{epoch}_model"
    model.save_pretrained(ckpt_path_epoch)
    print(f"saved checkpoint to {ckpt_path_epoch}")
    # checkpoint best
    if metrics["f1_score"] > best_f1:
        best_f1 = metrics["f1_score"]
        no_improve = 0
        ckpt_path = f"/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/models/best_round_{epoch}"
        model.save_pretrained(ckpt_path)
        print(f"  * new best! saved checkpoint to {ckpt_path}")
    else:
        no_improve += 1
        print(f"  * no improvement ({no_improve}/{patience})")

    # early‐stop
    if no_improve >= patience:
        print(f"\nNo improvement in {patience} successive rounds — stopping early.")
        break

    del metrics, detail_df, rev_results_processed, synthetic_results_processed, synthetic_res_this_batch, rev_results_this_batch
# ----------------------------------------
# Persist logs
# ----------------------------------------
history_df.to_csv("/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/active_learning_history.csv", index=False)
detailed_df_all.to_csv("/content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/detailed_holdout_results.csv", index=False)

print("\n=== Active‐Learning Complete ===")
print(f"Best holdout F1 over all rounds: {best_f1:.4f}")


=== Epoch 1: processing batch of 10 docs ===


loading batches:   0%|          | 0/10 [00:00<?, ?it/s]

infer on new batch:   0%|          | 0/792 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Revalidating using LLM:   0%|          | 0/100 [00:00<?, ?it/s]

...

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


training model on epoch 9


Holdout inference:   0%|          | 0/110 [00:00<?, ?it/s]

holdout metrics = {'tp': 250, 'fp': 20, 'fn': 20, 'precision': 0.9259259259259259, 'recall': 0.9259259259259259, 'f1_score': 0.9259259259259259}
    holdout F1 = 0.9259
saved checkpoint to /content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/models/round_9_model
  * new best! saved checkpoint to /content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/models/best_round_9

=== Active‐Learning Complete ===
Best holdout F1 over all rounds: 0.9259


In [ ]:
########################

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls -l /content/drive/MyDrive/colab-artifacts/refugee_data/REFUGEE_RE_POS/active_learning/models/

total 64
drwx------ 2 root root 4096 Jun 22 03:01 best_round_1
drwx------ 2 root root 4096 Jun 21 11:51 best_round_1_v2
drwx------ 2 root root 4096 Jun 22 03:30 best_round_2
drwx------ 2 root root 4096 Jun 20 16:18 best_round_4
drwx------ 2 root root 4096 Jun 22 04:39 best_round_5
drwx------ 2 root root 4096 Jun 22 05:00 best_round_6
drwx------ 2 root root 4096 Jun 22 06:05 best_round_9
drwx------ 2 root root 4096 Jun 22 02:59 round_1_model
drwx------ 2 root root 4096 Jun 22 03:28 round_2_model
drwx------ 2 root root 4096 Jun 22 03:55 round_3_model
drwx------ 2 root root 4096 Jun 22 04:17 round_4_model
drwx------ 2 root root 4096 Jun 22 04:39 round_5_model
drwx------ 2 root root 4096 Jun 22 05:00 round_6_model
drwx------ 2 root root 4096 Jun 22 05:23 round_7_model
drwx------ 2 root root 4096 Jun 22 05:46 round_8_model
drwx------ 2 root root 4096 Jun 22 06:05 round_9_model


In [ ]:
len(batches)

9